# ROBIN-FPGA Quickstart

This notebook walks through the simplest end-to-end use of ROBIN-FPGA:

1. Generate calibrated simulator data
2. Inspect the resulting CSV files
3. Build a quick figure preview

It does **not** require Vivado or Quartus — everything runs from the calibrated simulator.

**Estimated runtime:** under 30 seconds on a laptop.

## 1. Install (if you haven't already)

```bash
pip install -e ".[dev]"
```

In [ ]:
import sys, os
from pathlib import Path
# allow running from a checkout without `pip install -e .`
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import robin_fpga
print(f'ROBIN-FPGA version: {robin_fpga.__version__}')

## 2. Generate simulator data

The `Simulator` class produces all 13 paper figure CSV files in one call.

In [ ]:
from robin_fpga.simulator import Simulator, SimulatorConfig

sim = Simulator(SimulatorConfig(seed=42, output_dir='../data/results'))
paths = sim.generate_all()
for name, p in paths.items():
    print(f'{name:20s} -> {p}')

## 3. Inspect the closure-rate heatmap data

In [ ]:
import pandas as pd
df = pd.read_csv('../data/results/closure_rates.csv')
df.head()

In [ ]:
pivot = df.pivot(index='design', columns='baseline', values='closure_rate')
pivot

## 4. Plot the heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(pivot.columns)), pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)), pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                color='white' if v < 0.65 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, label='closure rate')
ax.set_title('Closure rate across designs and baselines')
fig.tight_layout()
plt.show()

## Next steps

- See `02_train_policy.ipynb` for training a fresh DR-PPO policy
- See `03_evaluation.ipynb` for the conformal sign-off procedure
- See `04_figure_generation.ipynb` to reproduce every paper figure